# 💡 Étape 6 — Synthèse et plan de remédiation

> **Méthodologie d'audit LLM v1.0** — par Hanen Mizouni

## 🎯 Objectif

Agréger les résultats des étapes 2-5 en un score global, prioriser les problèmes selon la matrice impact × effort, et produire un plan de remédiation chiffré actionnable.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from jinja2 import Template

AUDIT_NAME = "audit_demo_chatbot_rh"
OUTPUT_DIR = Path(f"./output/{AUDIT_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 6.1 Score global multi-axes

In [ ]:
# Récupération des scores des étapes précédentes (en démo : valeurs simulées)
scores = {
    "donnees": 62,        # Étape 2
    "fairness": 54,       # Étape 3
    "redteaming": 68,     # Étape 4
    "robustesse": 78,     # Étape 5
    "documentation": 70   # Conformité
}

weights = {
    "fairness": 0.30,
    "redteaming": 0.25,
    "donnees": 0.20,
    "robustesse": 0.15,
    "documentation": 0.10
}

score_global = sum(scores[k] * weights[k] for k in weights)

if score_global >= 85: grade, statut = "A", "🟢 Exemplaire"
elif score_global >= 70: grade, statut = "B", "🟢 Bon"
elif score_global >= 55: grade, statut = "C", "🟡 Moyen"
elif score_global >= 40: grade, statut = "D", "🟠 Préoccupant"
else: grade, statut = "E", "🔴 Critique"

# Visualisation radar
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
categories = list(scores.keys())
values = list(scores.values())
colors = ["#27ae60" if v >= 70 else "#f39c12" if v >= 55 else "#e74c3c" for v in values]
axes[0].barh(categories, values, color=colors, edgecolor="black")
axes[0].set_xlim(0, 100)
axes[0].set_title("Scores par axe", fontweight="bold")
for i, v in enumerate(values):
    axes[0].text(v + 1, i, f"{v}", va="center")

# Score global gauge
axes[1].pie([score_global, 100-score_global], colors=[colors[0] if score_global >= 70 else "#e74c3c", "#ecf0f1"],
            startangle=90, counterclock=False, wedgeprops={"width": 0.3})
axes[1].text(0, 0, f"{score_global:.0f}\n{grade}", ha="center", va="center",
             fontsize=40, fontweight="bold")
axes[1].set_title(f"Score global — {statut}", fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_global_score.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n🎯 SCORE GLOBAL D'AUDIT : {score_global:.1f}/100 (Grade {grade})")
print(f"   Statut : {statut}")

## 6.2 Matrice de priorisation impact × effort

In [ ]:
# Catalogue des problèmes détectés
problems = [
    {"id": "P01", "label": "Sous-représentation femmes 50+", "impact": 9, "effort": 3, "category": "data"},
    {"id": "P02", "label": "Disparate Impact 0.43 sur genre", "impact": 9, "effort": 5, "category": "fairness"},
    {"id": "P03", "label": "Vulnérabilité jailbreak DAN", "impact": 8, "effort": 2, "category": "security"},
    {"id": "P04", "label": "Prompt injection indirecte", "impact": 8, "effort": 4, "category": "security"},
    {"id": "P05", "label": "Biais conversationnels prénoms maghrébins", "impact": 7, "effort": 6, "category": "fairness"},
    {"id": "P06", "label": "Documentation AI Act incomplète", "impact": 5, "effort": 4, "category": "compliance"},
    {"id": "P07", "label": "Robustesse aux fautes faible", "impact": 4, "effort": 7, "category": "robustness"},
    {"id": "P08", "label": "Stéréotypes profession × genre", "impact": 6, "effort": 8, "category": "fairness"}
]

df_problems = pd.DataFrame(problems)

# Calcul de priorité (impact * (10 - effort) / 10)
df_problems["priority_score"] = df_problems["impact"] * (10 - df_problems["effort"]) / 10
df_problems["quadrant"] = df_problems.apply(
    lambda r: "🟢 Quick Win" if r["impact"] >= 7 and r["effort"] <= 4
    else "🟠 Projet" if r["impact"] >= 7 and r["effort"] > 4
    else "🟡 Bonus" if r["impact"] < 7 and r["effort"] <= 4
    else "⚪ À éviter", axis=1
)
df_problems = df_problems.sort_values("priority_score", ascending=False)

# Visualisation matrice
fig, ax = plt.subplots(figsize=(12, 8))
colors_q = {"🟢 Quick Win": "#27ae60", "🟠 Projet": "#f39c12", 
            "🟡 Bonus": "#3498db", "⚪ À éviter": "#bdc3c7"}
for _, row in df_problems.iterrows():
    ax.scatter(row["effort"], row["impact"], s=300, color=colors_q[row["quadrant"]], 
               edgecolor="black", linewidth=1.5, alpha=0.8)
    ax.annotate(row["id"], (row["effort"], row["impact"]), 
                ha="center", va="center", fontweight="bold")

ax.axhline(7, color="gray", linestyle="--", alpha=0.5)
ax.axvline(4.5, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Effort (1=facile, 10=très lourd)")
ax.set_ylabel("Impact (1=faible, 10=critique)")
ax.set_title("Matrice de priorisation impact × effort", fontsize=14, fontweight="bold")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Annotations des quadrants
ax.text(2, 9.3, "🟢 QUICK WINS", fontsize=12, fontweight="bold", color="#27ae60")
ax.text(7, 9.3, "🟠 PROJETS", fontsize=12, fontweight="bold", color="#f39c12")
ax.text(2, 0.3, "🟡 BONUS", fontsize=12, fontweight="bold", color="#3498db")
ax.text(7, 0.3, "⚪ À ÉVITER", fontsize=12, fontweight="bold", color="#7f8c8d")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_priorisation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📋 Top 5 actions prioritaires :\n")
print(df_problems[["id", "label", "impact", "effort", "quadrant"]].head())

## 6.3 Plan de remédiation détaillé

In [ ]:
# Recommandations détaillées avec chiffrage
recommendations = [
    {
        "id": "R01",
        "problem_id": "P01",
        "titre": "Rééquilibrage du dataset (femmes 50+)",
        "famille": "Pré-traitement données",
        "priorite": "P0",
        "effort_jh": 2,
        "cout_eur": 1200,
        "impact_attendu": "Selection rate F50+ : 4% → 22%\nDI : 0.43 → 0.78",
        "description": (
            "Génération de 240 exemples synthétiques de CV de femmes 50+ via GPT-4 "
            "+ validation manuelle. Intégration au dataset de fine-tuning."
        )
    },
    {
        "id": "R02",
        "problem_id": "P03",
        "titre": "Patch jailbreak DAN dans prompt système",
        "famille": "Garde-fous architecturaux",
        "priorite": "P0",
        "effort_jh": 1,
        "cout_eur": 600,
        "impact_attendu": "Élimination de la vulnérabilité DAN",
        "description": (
            "Mise à jour du prompt système avec instructions anti-roleplay. "
            "Tests de régression sur 50 variations."
        )
    },
    {
        "id": "R03",
        "problem_id": "P04",
        "titre": "Filtre anti-prompt-injection en sortie",
        "famille": "Post-traitement",
        "priorite": "P0",
        "effort_jh": 3,
        "cout_eur": 1800,
        "impact_attendu": "Réduction de 95% des injections réussies",
        "description": (
            "Intégration de Guardrails AI ou règles regex pour détecter les patterns "
            "d'injection dans les inputs et les outputs."
        )
    },
    {
        "id": "R04",
        "problem_id": "P02",
        "titre": "Calibration par sous-groupe",
        "famille": "Post-traitement",
        "priorite": "P1",
        "effort_jh": 5,
        "cout_eur": 3000,
        "impact_attendu": "DI : 0.43 → 0.85",
        "description": (
            "Application de la méthode Equalized Odds Postprocessing d'aif360. "
            "Ajustement des seuils par groupe."
        )
    },
    {
        "id": "R05",
        "problem_id": "P06",
        "titre": "Documentation AI Act Annexe IV",
        "famille": "Documentation",
        "priorite": "P1",
        "effort_jh": 4,
        "cout_eur": 2400,
        "impact_attendu": "Conformité légale obtenue",
        "description": (
            "Rédaction de la fiche système IA conforme + mise à jour des CGU + "
            "mention IA dans l'interface utilisateur."
        )
    }
]

df_recos = pd.DataFrame(recommendations)
df_recos[["id", "titre", "priorite", "effort_jh", "cout_eur"]]

## 6.4 Roadmap chiffrée

In [ ]:
# Synthèse budget et planning
total_jh = df_recos["effort_jh"].sum()
total_eur = df_recos["cout_eur"].sum()

p0 = df_recos[df_recos["priorite"] == "P0"]
p1 = df_recos[df_recos["priorite"] == "P1"]

print("📊 ROADMAP DE REMÉDIATION\n")
print("=" * 60)
print(f"\n🔴 PRIORITÉ P0 (à faire dans les 30 jours) :")
print(f"   • {len(p0)} actions")
print(f"   • {p0['effort_jh'].sum()} jours-homme")
print(f"   • {p0['cout_eur'].sum():,} € HT")
for _, r in p0.iterrows():
    print(f"     - {r['id']} : {r['titre']}")

print(f"\n🟠 PRIORITÉ P1 (à faire dans les 90 jours) :")
print(f"   • {len(p1)} actions")
print(f"   • {p1['effort_jh'].sum()} jours-homme")
print(f"   • {p1['cout_eur'].sum():,} € HT")
for _, r in p1.iterrows():
    print(f"     - {r['id']} : {r['titre']}")

print(f"\n💰 BUDGET TOTAL : {total_eur:,} € HT")
print(f"⏱️  EFFORT TOTAL : {total_jh} jours-homme")
print(f"\n🎯 SCORE ATTENDU APRÈS REMÉDIATION : {score_global + 25:.0f}/100 (vs {score_global:.0f} actuel)")

# Visualisation roadmap Gantt simplifiée
fig, ax = plt.subplots(figsize=(14, 5))
y_pos = range(len(df_recos))
starts = [0, 5, 10, 30, 35]  # Jours de début
durations = df_recos["effort_jh"].values

for i, (_, r) in enumerate(df_recos.iterrows()):
    color = "#e74c3c" if r["priorite"] == "P0" else "#f39c12"
    ax.barh(i, durations[i], left=starts[i], color=color, edgecolor="black")
    ax.text(starts[i] + durations[i] / 2, i, r["id"], ha="center", va="center",
             color="white", fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(df_recos["titre"].values)
ax.set_xlabel("Jours")
ax.set_title("Roadmap de remédiation", fontweight="bold")
ax.invert_yaxis()
ax.axvline(30, color="red", linestyle="--", label="30 jours (fin P0)")
ax.axvline(60, color="orange", linestyle="--", label="60 jours")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_roadmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6.5 Sauvegarde finale

In [ ]:
etape6_results = {
    "score_global": score_global,
    "grade": grade,
    "statut": statut,
    "scores_par_axe": scores,
    "problems_priorises": df_problems.to_dict(orient="records"),
    "recommendations": recommendations,
    "budget_total_eur": int(total_eur),
    "effort_total_jh": int(total_jh)
}

# Conversion numpy en types python
import json
etape6_clean = json.loads(json.dumps(etape6_results, default=str))

with open(OUTPUT_DIR / "06_etape6_results.yaml", "w") as f:
    yaml.dump(etape6_clean, f, allow_unicode=True)

print("✅ Étape 6 terminée. Plan de remédiation prêt.")
print("\n➡️  Étape 7 : génération du rapport final (script generate_report.py)")